# Healthcare Cost Analysis

This project analyzes a healthcare insurance dataset to identify the factors that influence individual medical costs.

The workflow uses **Tableau for exploratory data analysis (EDA)** and **Python for statistical modeling and machine learning**. Tableau was used to create the exploratory visualizations for relationships such as age vs. charges, BMI vs. charges, smoking status vs. charges, regional differences, and the cost distribution. Python was used for regression analysis, model evaluation, residual diagnostics, and decision tree feature importance.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score


## 2. Load the Dataset

In [ ]:
df = pd.read_csv('healthInsurance.csv')
df.head()

## 3. Inspect the Data

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

## 4. Data Preparation

Categorical variables are encoded so they can be used in regression and decision tree models.

In [ ]:
df_model = df.copy()
df_model['smoker'] = df_model['smoker'].map({'yes': 1, 'no': 0})
df_model['sex'] = df_model['sex'].map({'male': 1, 'female': 0})

# one-hot encode region so it can be included in regression cleanly
df_model = pd.get_dummies(df_model, columns=['region'], drop_first=True)
df_model.head()

## 5. Exploratory Data Analysis (Performed in Tableau)

The **exploratory data analysis for this project was performed in Tableau**, not Python.

The Tableau visuals used in this project include:

- Average healthcare cost by smoking status
- BMI vs. healthcare cost scatter plot
- Age vs. healthcare cost scatter plot
- Distribution of healthcare costs
- Average healthcare cost by region

These visualizations were used to identify the major relationships in the dataset before building predictive models. The most important exploratory finding was that **smoking status is the strongest driver of healthcare costs**, with age and BMI also showing meaningful relationships.

Python is still used below for a correlation heatmap and all model-based analysis.

## 6. Correlation Heatmap (Python)

Although the main EDA visuals were created in Tableau, this correlation heatmap is included in Python as a compact summary of linear relationships among the core numeric variables.

In [ ]:
corr_df = df.copy()
corr_df['smoker'] = corr_df['smoker'].map({'yes': 1, 'no': 0})
corr_df['sex'] = corr_df['sex'].map({'male': 1, 'female': 0})

plt.figure(figsize=(8, 6))
sns.heatmap(corr_df[['age', 'sex', 'bmi', 'children', 'smoker', 'charges']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Healthcare Cost Variables')
plt.tight_layout()
plt.show()

## 7. Linear Regression Model

In [ ]:
feature_columns = [
    'age', 'sex', 'bmi', 'children', 'smoker',
    'region_northwest', 'region_southeast', 'region_southwest'
]

X = df_model[feature_columns]
y = df_model['charges']

linear_model = LinearRegression()
linear_model.fit(X, y)
predictions = linear_model.predict(X)


## 8. Regression Metrics: R² and RMSE

In [ ]:
r2 = r2_score(y, predictions)
rmse = np.sqrt(mean_squared_error(y, predictions))

print(f'R²: {r2:.4f}')
print(f'RMSE: ${rmse:,.2f}')

### Interpretation

- **R²** shows how much of the variation in healthcare charges is explained by the regression model.
- **RMSE** shows the average prediction error in dollars.

Together, these metrics provide a practical summary of model fit and predictive accuracy.

## 9. Regression Coefficients

In [ ]:
coefficients = pd.Series(linear_model.coef_, index=X.columns).sort_values()
coefficients

In [ ]:
plt.figure(figsize=(8, 5))
coefficients.plot(kind='barh')
plt.title('Regression Coefficients for Healthcare Cost Prediction')
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()

## 10. Residual Analysis

In [ ]:
residuals = y - predictions

plt.figure(figsize=(8, 5))
plt.scatter(predictions, residuals, alpha=0.6)
plt.axhline(0, linestyle='--')
plt.xlabel('Predicted Charges')
plt.ylabel('Residuals')
plt.title('Residual Plot for Linear Regression Model')
plt.tight_layout()
plt.show()

### Residual Interpretation

The residual plot helps evaluate how well the linear regression model fits the data. If the spread increases at higher predicted values or shows structure, that suggests the relationship may not be fully linear and that a nonlinear model may provide additional insight.

## 11. Decision Tree Model

In [ ]:
tree_model = DecisionTreeRegressor(random_state=42, max_depth=4)
tree_model.fit(X, y)
tree_predictions = tree_model.predict(X)

tree_r2 = r2_score(y, tree_predictions)
tree_rmse = np.sqrt(mean_squared_error(y, tree_predictions))

print(f'Decision Tree R²: {tree_r2:.4f}')
print(f'Decision Tree RMSE: ${tree_rmse:,.2f}')

## 12. Feature Importance

In [ ]:
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': tree_model.feature_importances_
}).sort_values('Importance')

importance_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.xlabel('Importance Score')
plt.title('Decision Tree Feature Importance for Healthcare Cost Prediction')
plt.tight_layout()
plt.show()

## 13. Key Findings

- Exploratory data analysis was performed in **Tableau** to identify the major patterns in the dataset.
- **Smoking status** is the strongest predictor of healthcare costs.
- **Age** and **BMI** also contribute meaningfully to medical expenses.
- Regression metrics such as **R²** and **RMSE** provide evidence for model performance.
- The decision tree reinforces the importance of smoking status and helps capture nonlinear structure in the data.
